# Detector error models as tensor networks

This notebook computes probabilities of detector histories by contracting a tensor network built from a stim detector error model. Reading it top to bottom, the story is:

1. **The object.** A DEM defines a probability distribution over syndrome bit strings. We want to evaluate single entries of that distribution, $\Pr(x)$, without ever writing down the whole $2^N$-entry vector.
2. **The construction.** Each error mechanism becomes a small tensor, each detector becomes a parity constraint, and contracting the network performs the sum over all error configurations consistent with the observed syndrome. Section "What the contraction computes" gives the exact statement, in both the Blume-Kohout and Young language and the Derks et al. matrix language.
3. **Exact contraction** (`dem_tn.py`): validated against brute force, with observable legs for decoding, an admissibility check, and cached contraction paths.
4. **Approximate contraction** (`cluster_expansion.py`): belief propagation plus a connected cluster expansion, because exact contraction stops scaling. We measure exactly when and why the approximation works, including its honest failure mode.
5. **Applications**: reading off noise correlations from loop structure, and a first look at surface codes, where the network becomes three dimensional.
6. **Reproduction** (`experiments.py`): the final section rebuilds the summary figures used in the slide deck from module functions, and two command-line scripts (`run_surface_ce.py`, `run_correlated_validation.py`) run the same drivers outside the notebook.

Two papers run through everything. Blume-Kohout and Young (arXiv:2504.14643) supply the statistical language: events, attenuations, polarizations. Derks, Townsend-Teague, Burchards and Eisert (arXiv:2407.13826, "FTCircsWithDEMs") supply the linear-algebra language: detector matrices, error vectors, observables. The two descriptions are the same object viewed from different sides, and we translate between them as we go, because each makes different steps obvious.

In [53]:
# pip install these if necessary
import stim
import quimb as qu
import quimb.tensor as qtn
from itertools import product
import io
import numpy as np

In [54]:
def xor_tensor(num_vars, target_bit):
    # Construct a tensor that represents the xor operator
    shape = (2,) * num_vars
    data = []
    for bits in product([0, 1], repeat=num_vars):
        val = sum(bits) % 2
        data.append(1.0 if val == target_bit else 0.0) 
    return np.array(data).reshape(shape)

def stim_dem_to_tensor_network(dem: stim.DetectorErrorModel, target_bits):
    assert len(target_bits) == dem.num_detectors, "Number of target bits must equal number of detectors"
    tn = qtn.TensorNetwork([])
    detector_to_errors = {}
    error_index_map = {}  
    

    # tensor for each DEM event
    for i, event in enumerate(dem):
        prob = event.args_copy()[0]
        targets = [t.val for t in event.targets_copy() if t.is_relative_detector_id]
        main_ind = f'e{i}_main'
        prob_tensor = qtn.Tensor(
            data=[1 - prob, prob],
            inds=(main_ind,),
            tags={f'p{i}'}
        )
        tn |= prob_tensor

        aux_inds = [f'e{i}_{j}' for j in range(len(targets))]
        error_index_map[i] = aux_inds

        all_inds = [main_ind] + aux_inds
        id_tensor = np.zeros((2,) * len(all_inds))
        for bits in product([0, 1], repeat=len(all_inds)):
            if all(b == bits[0] for b in bits):
                id_tensor[bits] = 1.0
        tn |= qtn.Tensor(data=id_tensor, inds=all_inds, tags={f'e{i}'})

        for idx, d in enumerate(targets):
            detector_to_errors.setdefault(d, []).append((i, aux_inds[idx]))

    # XOR constraint tensors for each detector
    for d, target in enumerate(target_bits):
        if d not in detector_to_errors:
            if target != 0:
                return 0.0  # impossible
            continue

        involved = detector_to_errors[d]
        inds = [ind for (_, ind) in involved]
        t = xor_tensor(len(inds), target)
        tn |= qtn.Tensor(data=t, inds=inds, tags={f'd{d}'})

    return tn


def compute_probability_from_dem(dem: stim.DetectorErrorModel, target_bits):
    tn = stim_dem_to_tensor_network(dem, target_bits)
    return tn, tn.contract(all, optimize='auto-hq')


## What the contraction computes

Start with what a DEM actually is, because everything else follows from it. Stim hands us a list of error mechanisms. Each mechanism $i$ fires independently with probability $p_i$, and when it fires it flips a fixed set of detector bits. That is the whole model. In Blume-Kohout and Young's notation the mechanism is a bit string $\mathbf{s}_i$ marking which detectors it flips; in Derks et al. the same information is column $i$ of the detector error matrix $H$ (their Definition 2.9), and the firing pattern is a circuit error vector $\mathbf{e}$ (their Definition 2.6) drawn from the product of Bernoullis $\mathbf{p}_e$ (their Definition 2.4). The syndrome is then simply

$$\mathbf{x} = H\mathbf{e} \bmod 2,$$

which is Derks et al.'s $\mathbf{s} = H\mathbf{e}$, and identical to BK&Y's "matrix sampling recipe" $\mathbf{x} = E\mathbf{q}$ with $E = H$ and $\mathbf{q} = \mathbf{e}$. Same equation, two notations.

The quantity we want is the probability of seeing a particular syndrome:

$$\Pr(\mathbf{x}) \;=\; \sum_{\mathbf{e}\,:\,H\mathbf{e}=\mathbf{x}} \;\prod_i p_i^{e_i}(1-p_i)^{1-e_i}.$$

In words: add up the probability of every combination of error mechanisms that would have produced exactly this syndrome. Why is that hard? Because the sum runs over the fiber $H^{-1}(\mathbf{x})$, which contains $2^{L-\mathrm{rank}(H)}$ error vectors, exponentially many. And why do we want it anyway, instead of just running a decoder? Because this is the likelihood. A matching decoder or BP+OSD returns one plausible $\mathbf{e}$; this sum weighs the entire equivalence class. That distinction is exactly what Derks et al.'s Lemma 6 isolates as the invariant content of a DEM (different detector matrix choices distinguish the same error sets), and it is what a maximum-likelihood decoder, a model-validation test, or a Bayesian noise-inference loop actually needs.

The tensor network performs this sum without enumerating it. Three tensor types, one per ingredient of the formula:

- a **Bernoulli vector** $[1-p_i,\; p_i]$ for each mechanism: this is factor $i$ of $\Pr(\mathbf{e})$;
- a **COPY tensor** that broadcasts the mechanism's firing bit $e_i$ to every detector it touches: this is column $i$ of $H$, wired up as geometry instead of stored as a matrix;
- an **XOR (parity) tensor** at each detector, projected onto the observed bit $x_d$: this is row $d$ of the constraint $H\mathbf{e} = \mathbf{x}$.

Contracting the network sums over each $e_i$ locally, so the cost is governed by the contraction width of the network's graph (the largest intermediate tensor along the contraction order), not by $2^L$. For quasi-one-dimensional DEMs like the repetition code the width stays small at any number of rounds, which is why exact evaluation is cheap there. The whole game of this notebook is what happens to that width, and what to do when it grows.

One more identity worth knowing, because it connects to how BK&Y estimate DEMs from data. Write $d_i = 1-2p_i$ for each mechanism's decay factor. Then the Fourier transform of $\Pr(\mathbf{x})$ over detector bit strings has entries $\langle z_\mathbf{y}\rangle = \prod_i d_i^{\,\mathbf{y}\cdot\mathbf{s}_i}$, BK&Y's Eq. 23. Their paper inverts this to learn the $p_i$ from measured polarizations. Our network is the forward map of the same factorization: it takes a candidate set of $p_i$ and produces the probabilities those parameters imply. Learning and validation are the two directions of one arrow.

Sanity check on the toy DEM below: $\Pr([001]) = \Pr([101]) = 0.01902$. Those two histories differ by exactly one mechanism, the one with support $[100]$, so their ratio encodes its decay factor $d = 1-2p$, which is BK&Y's Eq. 19 doing its job in a three-detector example.

## Marginals, amplitudes, and cap vectors

Three questions tend to come up at this point, and they are really one question about what kind of computation a contraction is. Is $\Pr(\mathbf{x})$ a marginal? Is computing it an "observable computation" in the many-body sense? And what exactly happens at the open legs? This section settles all three.

**$\Pr(\mathbf{x})$ is a marginal, and the contraction is the marginalization.** The DEM's native joint distribution lives on the hidden mechanism bits together with everything they determine:

$$
\Pr(\mathbf{e}, \mathbf{x}, \ell) \;=\; \Pr(\mathbf{e})\,\cdot\, \mathbb{1}\!\left[\mathbf{x} = H\mathbf{e}\right]\,\cdot\, \mathbb{1}\!\left[\ell = O\mathbf{e}\right].
$$

Every internal index sum performed during contraction integrates out one latent bit $e_i$. So the contraction computes the marginal of this joint over all $2^L$ error configurations, and additionally over the logical outcome $\ell$ whenever we close the observable leg with the summing vector. The syndrome itself is never summed. It is clamped, which brings us to the mechanics.

**Cap vectors.** "Cap vector" is our informal name for the length-2 vector contracted into an open leg to close it (the standard operation is just contraction with a fixed boundary vector; the name is ours, not the literature's). Because every leg is one classical bit, there are only a few caps worth knowing, and together they generate every quantity in this notebook:

| cap on a leg | operation | meaning |
|---|---|---|
| $[1,\,0]$ or $[0,\,1]$ | clamp | evaluate at bit value $0$ or $1$ |
| $[1,\,1]$ | marginalize | sum the bit out (identity insertion) |
| $[1,\,-1]$ | Pauli-$Z$ insertion | weight configurations by $(-1)^{\text{bit}}$ |

Clamping every detector leg at the observed syndrome gives $\Pr(\mathbf{x})$ or, with the observable also clamped, $\Pr(\mathbf{x}, \ell)$. Replacing any clamp by $[1,1]$ marginalizes that detector. And the third row turns the network into an expectation-value engine, next paragraph.

**As a tensor network computation, $\Pr(\mathbf{x})$ is an amplitude evaluation, not an observable expectation.** The distinction matters when reading the many-body literature. There, a local expectation is a ratio of two closed networks, $\langle O\rangle = Z_O/Z$, a norm network and the same network with an operator inserted; the whole machinery of arXiv:2604.03228 (strings terminating on the operator region) exists to compute such ratios. Our computation is a single contraction with no denominator. The cleanest statement: the network is a factorized representation of Blume-Kohout and Young's $2^N$-entry probability vector,

$$
\vec P \;=\; \Big[\prod_{\mathbf{s}} L_{\mathbf{s}}\Big] \vec P_0 ,
\qquad
\Pr(\mathbf{x}) \;=\; \big\langle\, \mathbf{x} \,\big|\, \vec P \,\big\rangle,
$$

so clamping the detector legs reads off one component of a vector, structurally the same operation as evaluating a single amplitude $\langle x|\psi\rangle$ of a tensor-network state (the operation behind quantum-circuit amplitude sampling), and not the same operation as $\langle\psi|O|\psi\rangle$. There is also no doubled bra and ket layer anywhere, because a probability distribution is already the 1-norm object; classical distributions do not get squared. The reason no denominator is needed: the network is normalized by construction. Cap every leg with $[1,1]$ and it contracts to exactly $1$, each Bernoulli summing to one and each freed parity tensor factorizing, which is the analogue of $\langle\psi|\psi\rangle = 1$.

**The observable picture via caps.** Cap the legs of detectors in the support of $\mathbf{y}$ with $[1,-1]$ and all others with $[1,1]$, and the contraction returns a genuine expectation value under the distribution, the polarization

$$
\langle z_\mathbf{y}\rangle \;=\; \sum_{\mathbf{x}} \Pr(\mathbf{x})\,(-1)^{\mathbf{x}\cdot\mathbf{y}},
$$

in one contraction, again with no denominator, because $z_\mathbf{y}$ is diagonal in the syndrome basis. For a plain independent-event DEM this factorizes analytically, $\langle z_\mathbf{y}\rangle = \prod_s d_s^{\,\mathbf{y}\cdot\mathbf{s}}$ (BK&Y Eq. 23), so the network is overkill there; it starts doing real work once some legs are clamped to data or the model carries structure beyond independent events.

**Where a true ratio-of-networks computation does enter: conditionals.** A conditional is a ratio of marginals, and two of them matter here. The decoder's quantity is
$$
\Pr(\ell \,|\, \mathbf{x}) \;=\; \frac{\Pr(\mathbf{x}, \ell)}{\Pr(\mathbf{x})},
$$
two contractions or one open-leg contraction plus a normalization. The subtler one is the posterior probability that a specific mechanism fired,
$$
\Pr(e_i{=}1 \,|\, \mathbf{x}) \;=\; \frac{Z[\text{network with } e_i \text{ clamped to } 1]}{Z[\text{network unclamped}]},
$$
which is exactly the shape the local-observable formalism of arXiv:2604.03228 is built for: their expansion in clusters intersecting the clamped region computes precisely such ratios around a BP fixed point. This is the door through which calibrated soft decoder output (per-mechanism posteriors rather than syndrome likelihoods) would enter this project.

**None of this is specific to the repetition code, or to any code.** The construction consumes only the DEM abstraction, and Derks et al.'s point in building that abstraction is that any Clifford circuit compiles to one: surface, color, bivariate bicycle, Floquet codes, all become a list of mechanisms, a detector matrix and observable rows, and the caps act leg by leg with no knowledge of what code produced the leg. What changes between codes is geometry and multiplicity, not the formalism: the graph (hence contraction width and loop structure, as the surface-code section shows), and the number of logical observables, $k$ observable legs for a code with $k$ logical qubits, cappable independently or left open to return the full $2^k$-entry vector of joint logical-class probabilities in one contraction. That last point is worth registering: for a bivariate bicycle code with $k=12$, one contraction yields all $4096$ logical-class likelihoods, the object a matching decoder cannot produce at any price.

In [55]:
dem_str = """ 
error(0.01) D0
error(0.02) D0 D1
error(0.01) D1
error(0.02) D0 D2
error(0.02) D2
"""

dem_file = io.StringIO(dem_str)
dem = stim.DetectorErrorModel.from_file(dem_file)

for bits in product([0, 1], repeat=3):
    tn, prob = compute_probability_from_dem(dem, bits)
    print(bits, prob)


(0, 0, 0) 0.9224681599999999
(0, 0, 1) 0.01901984
(0, 1, 0) 0.009515839999999998
(0, 1, 1) 0.00058016
(1, 0, 0) 0.009892159999999999
(1, 0, 1) 0.01901984
(1, 1, 0) 0.018923839999999997
(1, 1, 1) 0.00058016


In [56]:
tree = tn.contraction_tree(optimize="greedy")
tree.plot_rubberband();

AttributeError: module 'matplotlib.cm' has no attribute 'get_cmap'

---

## The refactor: `dem_tn.py`

The prototype above works but has four gaps, each of which blocks a specific use. `dem_tn.py` closes them.

**Cheap tensors.** The prototype built its COPY and XOR tensors with Python loops over all $2^k$ index patterns, so a mechanism or detector of degree $k$ cost $2^k$ memory just to construct. That is fine at degree 5 and fatal at degree 48 (we will meet degree 48 in the surface-code section). The fix is structural: `qtn.COPY_tensor` for the broadcasts, and for any parity node above degree 8 an exact decomposition into a chain of degree-3 XOR tensors carrying a running parity. This works because XOR is associative: checking $i_1\oplus i_2\oplus\cdots\oplus i_k = b$ in one shot or by accumulating pairwise gives the same constraint, but the chain costs $O(k)$ memory instead of $O(2^k)$.

**Observable legs.** Derks et al.'s Definition 2.10 makes the observable a parity of measurements, algebraically just another row, $\mathbf{o}^T\Omega$, sitting alongside the detector rows. So we treat it as another parity node in the network. Project its leg onto $\ell$ and the contraction returns the joint probability $\Pr(\mathbf{x}, L{=}\ell)$; leave the leg open and one contraction returns the whole vector over $\ell$. This is what turns the evaluator into a maximum-likelihood decoder: their Eq. 24 declares a logical error by comparing the observable parity of the true error against the decoder's inference, and the error class that maximizes $\Pr(\mathbf{x}, L{=}\ell)$ is by definition the best possible inference. A matching decoder approximates this; the contraction computes it.

**Admissibility.** Some syndromes cannot be produced by any combination of mechanisms: precisely those outside the column span of $H$. For them $\Pr(\mathbf{x})$ is exactly zero, and asking the contraction to discover that zero numerically is asking for underflow and wasted work. A GF(2) rank check, $\mathrm{rank}(H\,|\,\mathbf{x}) > \mathrm{rank}(H)$ means inadmissible, settles it before any tensor is built. This is also the natural home for the prototype's `return 0.0` special case, which only caught the trivial subcase of a detector with no incident mechanisms.

**Path caching.** Finding a good contraction order is itself expensive (cotengra searches over orders), but every syndrome for a fixed DEM produces a network with identical shape; only the numbers inside the XOR projections change. So the search is paid once and reused across thousands of syndromes via a shared `ReusableHyperOptimizer`. When a later cell seems to stall on its first contraction, it is this one-time search you are watching.

Validation below is against brute-force enumeration, which is slow but unarguable: the toy DEM with an observable (both projected and open leg), and a real stim circuit-level repetition-code DEM at $d{=}3$, $R{=}3$, all 512 admissible syndrome and observable combinations. Agreement is at the $10^{-15}$ level, i.e. floating-point exact.

In [ ]:
from dem_tn import DemTN, contract_exact, make_optimizer, contract_bp_gloop_near_observable

# --- validate on the toy DEM, now with an observable added to two events ---
dem_str_obs = """
error(0.01) D0 L0
error(0.02) D0 D1
error(0.01) D1 L0
error(0.02) D0 D2
error(0.02) D2
"""
dem_obs = stim.DetectorErrorModel.from_file(io.StringIO(dem_str_obs))
d = DemTN(dem_obs)


def brute_force(dem, syndrome, obs_val):
    total = 0.0
    for bits in product([0, 1], repeat=dem.num_errors):
        p = 1.0
        det_parity = [0] * dem.num_detectors
        obs_parity = [0] * dem.num_observables
        for b, ev in zip(bits, dem):
            pe = ev.args_copy()[0]
            p *= pe if b else (1 - pe)
            for t in ev.targets_copy():
                if t.is_relative_detector_id():
                    det_parity[t.val] ^= b
                if t.is_logical_observable_id():
                    obs_parity[t.val] ^= b
        if det_parity == list(syndrome) and obs_parity == [obs_val]:
            total += p
    return total


max_err = 0.0
for bits in product([0, 1], repeat=3):
    for ell in (0, 1):
        tn = d.build(bits, observables={0: ell})
        val = 0.0 if tn is None else contract_exact(tn)
        max_err = max(max_err, abs(val - brute_force(dem_obs, bits, ell)))
print(f"projected-observable max abs err vs brute force: {max_err:.3e}")

# open-leg mode: don't project the observable, get both entries from one contraction
tn = d.build((0, 0, 1), observables={})
open_ind = d.obs_tags[0]  # 'obs0' -- the only index appearing on exactly 1 tensor
res = contract_exact(tn, output_inds=(open_ind,))
vec = res.data
bf = [brute_force(dem_obs, (0, 0, 1), ell) for ell in (0, 1)]
print(f"open-leg vec {vec} vs brute force {bf}, max err {max(abs(vec[0]-bf[0]), abs(vec[1]-bf[1])):.3e}")


projected-observable max abs err vs brute force: 2.220e-16
open-leg vec [0.0188258  0.00019404] vs brute force [0.0188258, 0.00019403999999999998], max err 3.469e-18


In [ ]:
# --- validate on a real circuit-level DEM: d=3, R=3 repetition code ---
circuit = stim.Circuit.generated(
    "repetition_code:memory",
    rounds=3,
    distance=3,
    before_round_data_depolarization=0.03,
    before_measure_flip_probability=0.01,
)
dem_rep = circuit.detector_error_model(decompose_errors=False)
print("num_detectors:", dem_rep.num_detectors, "num_observables:", dem_rep.num_observables,
      "num_events:", dem_rep.num_errors)

d_rep = DemTN(dem_rep)


def brute_force_all(dem):
    dem = dem.flattened()
    events = [ev for ev in dem if ev.type == "error"]
    probs = {}
    for bits in product([0, 1], repeat=len(events)):
        p = 1.0
        det_parity = [0] * dem.num_detectors
        obs_parity = [0] * dem.num_observables
        for b, ev in zip(bits, events):
            pe = ev.args_copy()[0]
            p *= pe if b else (1 - pe)
            if p == 0.0:
                break
            for t in ev.targets_copy():
                if t.is_relative_detector_id():
                    det_parity[t.val] ^= b
                if t.is_logical_observable_id():
                    obs_parity[t.val] ^= b
        else:
            key = (tuple(det_parity), tuple(obs_parity))
            probs[key] = probs.get(key, 0.0) + p
    return probs


bf_rep = brute_force_all(dem_rep)  # 2**18 configurations, a few seconds
opt = make_optimizer(max_repeats=32)  # one path search, reused for every syndrome below
max_err = 0.0
for (syn, obs), bf_val in bf_rep.items():
    tn = d_rep.build(syn, observables={0: obs[0]})
    val = 0.0 if tn is None else contract_exact(tn, optimize=opt)
    max_err = max(max_err, abs(val - bf_val))
print(f"checked {len(bf_rep)} admissible (syndrome, observable) pairs; max abs err = {max_err:.3e}")


num_detectors: 8 num_observables: 1 num_events: 18
checked 512 admissible (syndrome, observable) pairs; max abs err = 3.997e-15


---

## First attempt at approximation: BP plus a localized loop series

Why approximate at all? Exact contraction cost grows exponentially in the contraction width, and width grows with distance and code dimensionality. The literature's answer is belief propagation: treat the network as if it had no loops, pass messages until they stop changing, and read off an estimate. BP is exact on trees and cheap everywhere, but our networks have loops, and everything BP gets wrong lives in them.

This section is a first, flawed attempt at correcting BP, kept in the notebook because the way it fails is instructive. Quimb ships two loop-correction routines and they are built on different mathematics:

- `HD1BP.contract_gloop_expand` implements Kikuchi-style region counting. Its inclusion-exclusion coefficients only cancel correctly when the region set covers the network completely. Feed it a handful of loops near one tensor and the counting is simply wrong; in our tests it returned probabilities near 1 regardless of the true value.
- `D1BP.contract_loop_series_expansion` implements the loop series of Evenbly, Pancotti, Milsted, Gray and Chan (arXiv:2409.03108), where each loop adds an independent term. A partial set of loops is then a legitimate partial correction, which is why this cell uses it, restricted to loops grown from the observable's tensor, on the reasoning that the decoding decision is made there.

One structural remark that makes any of this possible: although the DEM has genuine hyperedges (mechanisms flipping three or more detectors), our network is an ordinary graph, because the builder materializes every broadcast as an explicit COPY tensor. Every index connects exactly two tensors, so plain `D1BP` applies with no hypergraph preprocessing.

The numbers below show a real but modest gain on the likely observable value and much less on the rare one. The next section explains what this approach gets structurally wrong and replaces it.

In [ ]:
from quimb.tensor.belief_propagation import D1BP

circuit5 = stim.Circuit.generated(
    "repetition_code:memory",
    rounds=5,
    distance=5,
    before_round_data_depolarization=0.05,
    before_measure_flip_probability=0.02,
)
dem5 = circuit5.detector_error_model(decompose_errors=False)
d5 = DemTN(dem5)
print("num_detectors:", dem5.num_detectors, "num_events:", d5.num_events)

sampler = circuit5.compile_detector_sampler(seed=0)
dets, obs = sampler.sample(1, separate_observables=True)
syn = dets[0].astype(int)
true_obs = int(obs[0][0])
print(f"sampled syndrome weight {syn.sum()}, true observable L={true_obs}")

opt5 = make_optimizer(max_repeats=32)
for ell in (true_obs, 1 - true_obs):
    tn = d5.build(syn, observables={0: ell})
    exact_val = contract_exact(tn, optimize=opt5)
    print(f"\nL={ell} ({'true' if ell == true_obs else 'wrong'} branch), exact = {exact_val:.6e}")
    for max_size in (2, 4, 6, 8):
        bp, gloops, val = contract_bp_gloop_near_observable(tn.copy(), obs_tag="obs0", max_size=max_size)
        bp_val = bp.contract()
        print(f"  max_size={max_size:2d}  n_gloops={len(gloops):3d}  "
              f"BP={bp_val:.6e} (err {abs(bp_val-exact_val):.2e})   "
              f"BP+gloop={val:.6e} (err {abs(val-exact_val):.2e})")


num_detectors: 24 num_events: 50
sampled syndrome weight 0, true observable L=0

L=0 (true branch), exact = 2.586258e-01
  max_size= 2  n_gloops=  0  BP=2.585948e-01 (err 3.11e-05)   BP+gloop=2.585948e-01 (err 3.11e-05)
  max_size= 4  n_gloops=  0  BP=2.585948e-01 (err 3.11e-05)   BP+gloop=2.585948e-01 (err 3.11e-05)
  max_size= 6  n_gloops=  5  BP=2.585948e-01 (err 3.11e-05)   BP+gloop=2.586235e-01 (err 2.29e-06)
  max_size= 8  n_gloops=  9  BP=2.585948e-01 (err 3.11e-05)   BP+gloop=2.586240e-01 (err 1.83e-06)

L=1 (wrong branch), exact = 7.350240e-08
  max_size= 2  n_gloops=  0  BP=2.423627e-03 (err 2.42e-03)   BP+gloop=2.423627e-03 (err 2.42e-03)
  max_size= 4  n_gloops=  0  BP=2.423627e-03 (err 2.42e-03)   BP+gloop=2.423627e-03 (err 2.42e-03)
  max_size= 6  n_gloops=  5  BP=2.423627e-03 (err 2.42e-03)   BP+gloop=1.101251e-03 (err 1.10e-03)
  max_size= 8  n_gloops=  9  BP=2.423627e-03 (err 2.42e-03)   BP+gloop=1.210169e-03 (err 1.21e-03)


---

## The right correction: a connected cluster expansion (`cluster_expansion.py`)

The loop series above corrects the partition function $Z$ directly, and Midha and Zhang (arXiv:2510.02290) prove that this is the wrong object to expand. Their argument is worth internalizing because it is physical, not technical. $Z$ is multiplicative: perturb one site and $Z$ changes by a factor, so any additive series for $Z$ must include terms touching every combination of sites, and the number of disconnected loop combinations grows combinatorially, faster than individual loop contributions shrink. The series diverges by construction. The free energy $\log Z$ is additive: perturb one site and $\log Z$ shifts by a constant, so its series only needs connected objects, and the count of connected clusters grows merely exponentially, slow enough for exponentially decaying loop terms to beat. That single observation converts a divergent expansion into one with a proof of exponential convergence (their Theorem III.1, given loop decay $|Z_l| \le e^{-c|l|}$ with $c$ above a threshold set by the graph degree).

`cluster_expansion.py` implements their construction directly from the paper, using quimb only for the BP fixed point and the tensor plumbing:

- **Loops** are connected edge subsets in which every vertex has at least two incident subset edges (their Definition II.1). This is enumerated by hand, by growing edge sets breadth-first with their pruning rule, rather than with quimb's `gen_gloops`, because quimb enumerates vertex sets and the paper's excitations are edge sets; the distinction matters when a vertex set admits several edge configurations.
- **Loop corrections** $Z_l$ insert the excitation projector $P^\perp = \mathbb{1} - |\mu_{w\to v}\rangle\langle\mu_{v\to w}|$ on each loop edge and close every other leg with its incoming BP message, on tensors normalized so the BP value is 1. The projector removes exactly the part of the bond that BP already accounted for, so $Z_l$ is the part of the answer BP missed through that loop, which is the reason these objects are the right correction currency.
- **Clusters** are multisets of loops, kept only when connected through the interaction graph of pairwise overlaps, weighted by the Ursell function $\phi(\mathbf{W})$ (their Eq. 16), which we evaluate by brute-force inclusion-exclusion over spanning connected subgraphs. It is affordable because relevant clusters contain few loops.
- The result assembles as $Z \approx Z_{BP}\, e^{\tilde F_m}$ with $\tilde F_m = \sum_{|\mathbf{W}|\le m} \phi(\mathbf{W}) Z_{\mathbf{W}}$.

Correctness checks, chosen because each has an independently known answer: a single-ring network reproduces $Z_{BP}(1+Z_l)$ to machine precision and its cluster series recovers the Taylor expansion of $\log(1+Z_l)$ term by term, which is the paper's own Section III.C example; and a $4\times4$ classical Ising network drops from BP error $7\times10^{-2}$ to $1\times10^{-5}$ at truncation weight 12, monotonically, matching the qualitative shape of their Figure 4. Neither of quimb's built-in routines computes this object, which is why the earlier section behaved as it did.

In [ ]:
from cluster_expansion import ClusterExpansion

# --- DEM convergence vs cluster weight m, incl. the regime where quimb's
# --- built-in gloop machinery diverged (d=3, R=4, p=0.35)
circuit_a = stim.Circuit.generated(
    "repetition_code:memory", rounds=4, distance=3,
    before_round_data_depolarization=0.35, before_measure_flip_probability=0.35,
)
dem_a = circuit_a.detector_error_model(decompose_errors=False)
d_a = DemTN(dem_a)
sampler_a = circuit_a.compile_detector_sampler(seed=1)
dets_a, _ = sampler_a.sample(1, separate_observables=True)
syn_a = dets_a[0].astype(int)

tn_a = d_a.build(syn_a, observables=None)
tn_a |= qtn.Tensor(np.ones(2), inds=("obs0",))  # marginalize observable
opt_a = make_optimizer(max_repeats=32)
exact_a = contract_exact(tn_a, optimize=opt_a)
print(f"d=3 R=4 p=0.35, syndrome weight {syn_a.sum()}: exact Pr(x) = {exact_a:.6e}")

ce = ClusterExpansion(tn_a)
bp = ce.contract_bp()
print(f"  BP  (m=0): rel_err = {abs(bp-exact_a)/exact_a:.3e}")
for m in (6, 8, 10):
    val, info = ce.contract(m, return_info=True)
    print(f"  m={m:2d}: rel_err = {abs(val-exact_a)/exact_a:.3e}  "
          f"[{info['n_loops']} loops, {info['n_clusters']} clusters]")


d=3 R=4 p=0.35, syndrome weight 4: exact Pr(x) = 9.854652e-04
  BP  (m=0): rel_err = 3.478e-03
  m= 6: rel_err = 3.478e-03  [4 loops, 4 clusters]
  m= 8: rel_err = 3.791e-04  [11 loops, 11 clusters]
  m=10: rel_err = 3.791e-04  [20 loops, 20 clusters]


In [ ]:
# --- d=5, R=5: cluster expansion on the decoding branches, and
# --- the loop-decay ('c-decay') diagnostic vs physical error rate
for ell in (true_obs, 1 - true_obs):
    tn5 = d5.build(syn, observables={0: ell})
    exact5 = contract_exact(tn5, optimize=opt5)
    print(f"L={ell} ({'true' if ell == true_obs else 'wrong'} branch): exact = {exact5:.6e}")
    ce5 = ClusterExpansion(tn5)
    bp5 = ce5.contract_bp()
    print(f"  BP  (m=0): rel_err = {abs(bp5-exact5)/abs(exact5):.3e}")
    for m in (6, 8):
        val5 = ce5.contract(m)
        print(f"  m={m}: rel_err = {abs(val5-exact5)/abs(exact5):.3e}")
    print()

# loop-decay: max |Z_l| per loop weight, sweeping p (d=3, R=3, trivial syndrome)
from collections import defaultdict

print("c_eff(|l|) = -log(max |Z_l|)/|l|   (convergence needs c_eff > c_0 ~ log Delta)")
opt_c = make_optimizer(max_repeats=32)
for p in (0.01, 0.05, 0.15, 0.30, 0.45):
    circ = stim.Circuit.generated(
        "repetition_code:memory", rounds=3, distance=3,
        before_round_data_depolarization=p, before_measure_flip_probability=p / 2,
    )
    dm = DemTN(circ.detector_error_model(decompose_errors=False))
    tnc = dm.build(np.zeros(dm.num_detectors, dtype=int), observables=None)
    tnc |= qtn.Tensor(np.ones(2), inds=("obs0",))
    exact_c = contract_exact(tnc, optimize=opt_c)
    cec = ClusterExpansion(tnc)
    by_w = defaultdict(list)
    for F in cec.gen_loops(10):
        by_w[len(F)].append(abs(cec.loop_correction(F)))
    decay = "  ".join(f"|l|={w}: {max(v):.1e}" for w, v in sorted(by_w.items()))
    errs = "  ".join(f"m={m}: {abs(cec.contract(m)-exact_c)/exact_c:.1e}" for m in (6, 8))
    print(f"p={p:.2f}  BP err {abs(cec.contract_bp()-exact_c)/exact_c:.1e}  {errs}   max|Z_l|: {decay}")


L=0 (true branch): exact = 2.586258e-01
  BP  (m=0): rel_err = 1.202e-04
  m=6: rel_err = 8.758e-06
  m=8: rel_err = 1.623e-07

L=1 (wrong branch): exact = 7.350240e-08


/Users/aswathsurya/Desktop/workspaces/workspace1/VSCodeFiles_Thinkpad/.venv/lib/python3.12/site-packages/quimb/tensor/belief_propagation/bp_common.py:373: UserWarning: Belief propagation did not converge after 1000 iterations, tol=5.00e-09, max|dM|=1.61e-06.
  warnings.warn(


  BP  (m=0): rel_err = 3.297e+04
  m=6: rel_err = 1.910e+04
  m=8: rel_err = 1.998e+04

c_eff(|l|) = -log(max |Z_l|)/|l|   (convergence needs c_eff > c_0 ~ log Delta)
p=0.01  BP err 3.1e-09  m=6: 3.1e-09  m=8: 5.1e-14   max|Z_l|: |l|=6: 2.1e-39  |l|=8: 1.1e-09  |l|=10: 9.6e-44
p=0.05  BP err 2.1e-06  m=6: 2.1e-06  m=8: 8.9e-10   max|Z_l|: |l|=6: 5.0e-37  |l|=8: 7.8e-07  |l|=10: 5.9e-40
p=0.15  BP err 2.1e-04  m=6: 2.1e-04  m=8: 8.3e-07   max|Z_l|: |l|=6: 5.6e-36  |l|=8: 7.6e-05  |l|=10: 6.7e-38
p=0.30  BP err 3.3e-03  m=6: 3.3e-03  m=8: 5.6e-05   max|Z_l|: |l|=6: 5.0e-35  |l|=8: 1.2e-03  |l|=10: 9.6e-36
p=0.45  BP err 5.4e-03  m=6: 5.4e-03  m=8: 2.4e-04   max|Z_l|: |l|=6: 1.6e-36  |l|=8: 2.0e-03  |l|=10: 4.0e-35


### What the numbers above are telling us

**Convergence is real and exponential.** On the noisy $d{=}3$, $R{=}4$, $p{=}0.35$ DEM, where quimb's region-counting routine produced errors in the thousands, the cluster expansion moves monotonically from BP's $3.5\times10^{-3}$ to $3.8\times10^{-4}$ at weight 8. On the $d{=}5$, $R{=}5$ likely branch it goes $1.2\times10^{-4} \to 8.8\times10^{-6} \to 1.6\times10^{-7}$ across weights 0, 6, 8. Each added weight class buys roughly a fixed factor, which is what "exponential in $m$" looks like in a table.

**Why weights 6 and 10 contribute nothing here.** Two separate reasons, and it took a per-tensor dissection of $Z_l$ to pin them down (possible because with bond dimension 2 every $P^\perp$ is rank one, so a cycle's value is a literal product of per-tensor numbers, and one can ask which factor vanishes).

First reason: counting. The network alternates mechanism tensors and detector tensors, so a cycle of $k$ mechanisms has weight $2k$. A weight-6 loop is therefore a triangle of three mechanisms in the detector graph. But the repetition code's detector graph is a grid: space edges from data errors, time edges from measurement errors, and a grid has no odd cycles. Triangles and pentagons cannot exist, so the first genuine cycles are grid plaquettes of four mechanisms, weight 8.

Second reason: the observable. The few weight-6 candidates that do exist route through the observable node, since several mechanisms share $L_0$ and it acts as an extra shared vertex. We marginalized the observable by capping its parity tensor with $[1,1]$, and summing a parity constraint over both outcomes deletes the constraint. What remains is the all-ones tensor, rank one, and its single direction is exactly the message BP sends through it. The excitation projector is built to annihilate that direction, so the loop factor is zero. Not small: zero, and the dissection shows it at $10^{-32}$, pure float dust. Two cross-checks make this airtight. Project the observable instead of marginalizing and those loops come alive, which is why the decoding runs improve already at weight 6. And the injected correlated events of the next section create genuine triangles, whose weight-6 loops carry $10^{-4}$.

**Noise rate behaves like temperature.** The decay rate $c_{\rm eff} = -\log(\max|Z_l|)/|l|$ of the weight-8 class falls from 2.6 at $p{=}0.01$ to 0.78 at $p{=}0.45$. This mirrors the Ising benchmark in the cluster-expansion papers, where loop decay weakens on approach to the critical temperature. The practical reading: the noisier the device, the more loop weight you need for a given accuracy, and there will be a noise level beyond which the expansion stops converging at all.

**The failure mode, stated plainly.** On the wrong-observable branch the true probability is $7\times10^{-8}$ and BP converges to $2.4\times10^{-3}$, four orders too high, and no truncation weight we can afford repairs it. This is the fixed-point problem of Midha, Sommers, Tindall and Abanin (arXiv:2604.03228, Section II.F): the cluster series converges to the answer implied by the fixed point you expand around, and a bad mean field cannot be fixed by decorating it. For picking the more likely observable this shot survives, since $2.4\times10^{-3}$ is still far below the true branch's $0.26$. But that margin is an artifact of the fixed point, not a computed likelihood, so any use that needs calibrated tail probabilities, soft decoding information above all, still requires exact contraction on the rare branch.

---

## Correlated noise, and why loops are where it lives

The question this section answers: if two distant parts of the device fail together (crosstalk, a shared control line, a cosmic ray), can the tensor network tell, and specifically can the loop corrections tell? The motivation comes straight from the opening of Blume-Kohout and Young's paper: a decoder looking at one shot cannot distinguish "qubits 1 and 2 flipped coincidentally" from "one mechanism flipped both", because the two produce identical syndromes shot by shot. The difference is statistical, and it is entirely a difference in correlations.

To make that razor sharp, `with_correlated_events` builds a matched pair of models. The correlated model $\mathcal{D}^*$ appends a mechanism `error(p_c) D_i D_j` that flips two chosen detectors together. The decorrelated model $\mathcal{D}'$ instead appends two independent single-detector mechanisms at the same rate. Because attenuations add when mechanisms combine (BK&Y Eq. 22), every single-detector statistic $\langle z_i\rangle$ comes out identical between the two models, and the cell verifies this to the last digit. Whatever distinguishes them is therefore correlation and nothing else. In Derks et al. terms both models share the same single-column marginals of $H\mathbf{e}$; they differ only in whether two rows of $H$ share a column.

Where does the pair go in the network? We place the correlated pairs at spatial distance two inside a round, between detectors already linked through existing mechanisms. That choice is deliberate: a correlated event between otherwise disconnected detectors would just be a tree branch, BP handles trees exactly, and nothing interesting would happen. Placed inside connected territory, the new mechanism closes new short loops, and loops are exactly what BP cannot see. The prediction from Midha, Sommers, Tindall and Abanin's Proposition IV.1 (connected correlations are carried entirely by clusters connecting the regions) is then concrete: the correlation should appear as specific loop corrections and BP should partly miss it.

Both predictions check out in the cells below. The loop spectrum shows new weight-6 loops through the injected mechanism carrying $|Z_l| \sim 10^{-4}$, against a background of $10^{-37}$ at that weight, because the injected event creates a triangle in a detector graph that is otherwise bipartite and cannot have triangles at all. And the likelihood-ratio test, sampling shots from $\mathcal{D}^*$ and computing $\log \Pr_{\mathcal{D}^*}(x) - \log\Pr_{\mathcal{D}'}(x)$ per shot, gives a mean of $0.145$ nats, which is the Kullback-Leibler information the correlation carries per shot. BP underestimates that information by 14 percent on average and by about 37 percent on precisely the shots that light up the correlated detectors, because its tree approximation partially factorizes the correlation away. The cluster expansion at weight 8 recovers the ratio to under one percent. The conclusion to take forward: a BP-only likelihood systematically under-detects cross-site correlation, and the loop corrections are not a numerical refinement on top of the answer, they are where the correlation information is stored.

In [ ]:
from dem_tn import with_correlated_events

P_C = 0.08
circuit_x = stim.Circuit.generated(
    "repetition_code:memory", rounds=4, distance=4,
    before_round_data_depolarization=0.05, before_measure_flip_probability=0.02,
)
dem0 = circuit_x.detector_error_model(decompose_errors=False)
corr_events = [(P_C, [3, 5]), (P_C, [6, 8])]  # distance-2 pairs within a round
dem_c, dem_d = with_correlated_events(dem0, corr_events)
dc, dd = DemTN(dem_c), DemTN(dem_d)
corr_ids = list(range(dc.num_events - len(corr_events), dc.num_events))
opt_x = make_optimizer(max_repeats=32)


def marginal_tn(d, syn):
    tn = d.build(syn, observables=None)
    tn |= qtn.Tensor(np.ones(2), inds=("obs0",))
    return tn


# 1. marginal check: single-detector polarizations identical by construction
def pol(d, i):  # <z_i> = prod over events flipping i of (1 - 2p), BK&Y Eq. 23
    z = 1.0
    for j in range(d.num_events):
        if i in d.event_dets[j]:
            z *= 1 - 2 * d.probs[j]
    return z

for i in (3, 5, 6, 8):
    print(f"<z_{i}>: corr={pol(dc, i):+.6f}  decorr={pol(dd, i):+.6f}  "
          f"diff={abs(pol(dc, i) - pol(dd, i)):.1e}")

# 2. loop spectrum: loops through the correlated events vs elsewhere
syn0 = np.zeros(dem0.num_detectors, dtype=int)
prefixes = tuple(f"e{j}_" for j in corr_ids)
for name, d in (("corr", dc), ("decorr", dd)):
    ce = ClusterExpansion(marginal_tn(d, syn0))
    by = defaultdict(lambda: [0.0, 0.0])
    for F in ce.gen_loops(8):
        zl = abs(ce.loop_correction(F))
        k = 0 if any(ix.startswith(prefixes) for ix in F) else 1
        by[len(F)][k] = max(by[len(F)][k], zl)
    for w in sorted(by):
        print(f"[{name}] |l|={w}: max|Z_l| through corr-event = {by[w][0]:.2e},"
              f" elsewhere = {by[w][1]:.2e}")

# 3. per-shot LLR (exact / BP / cluster m=8), shots from the CORRELATED model
dets_x, obs_x, _ = dem_c.compile_sampler(seed=7).sample(shots=12)
rows = []
for k in range(dets_x.shape[0]):
    syn = dets_x[k].astype(int)
    if not dc.is_admissible(syn):
        continue
    v = {}
    for name, d in (("corr", dc), ("decorr", dd)):
        tn = marginal_tn(d, syn)
        ce = ClusterExpansion(tn)
        v[name] = (contract_exact(tn, optimize=opt_x), ce.contract_bp(), ce.contract(8))
    rows.append([np.log(v["corr"][j]) - np.log(v["decorr"][j]) for j in range(3)])
rows = np.array(rows)
print(f"\nmean LLR over {len(rows)} shots:  exact={rows[:,0].mean():+.4f}  "
      f"BP={rows[:,1].mean():+.4f}  cluster m=8={rows[:,2].mean():+.4f}")
print(f"mean |LLR error| vs exact:  BP={np.abs(rows[:,1]-rows[:,0]).mean():.2e}  "
      f"cluster m=8={np.abs(rows[:,2]-rows[:,0]).mean():.2e}")


<z_3>: corr=+0.674365  decorr=+0.674365  diff=0.0e+00
<z_5>: corr=+0.674365  decorr=+0.674365  diff=0.0e+00
<z_6>: corr=+0.674365  decorr=+0.674365  diff=0.0e+00
<z_8>: corr=+0.674365  decorr=+0.674365  diff=0.0e+00
[corr] |l|=6: max|Z_l| through corr-event = 1.03e-04, elsewhere = 9.57e-38
[corr] |l|=8: max|Z_l| through corr-event = 3.13e-06, elsewhere = 4.94e-07
[decorr] |l|=6: max|Z_l| through corr-event = 0.00e+00, elsewhere = 3.76e-37
[decorr] |l|=8: max|Z_l| through corr-event = 0.00e+00, elsewhere = 4.87e-07

mean LLR over 12 shots:  exact=+0.1449  BP=+0.1338  cluster m=8=+0.1457
mean |LLR error| vs exact:  BP=2.07e-02  cluster m=8=9.49e-04


---

## Visualizations (`dem_viz.py`)

Each figure carries one claim from the sections above, with a fixed color language so nothing has to be relearned between plots: blue is always the cluster expansion or the correlated model, orange is always BP or the decorrelated model, gray is context. Every figure is also written to `figures/` as a PNG when its cell runs, so the results survive kernel restarts.

1. `plot_convergence`: error of $\Pr(x)$ against truncation weight, log scale, the exponential-convergence claim as a picture.
2. `plot_loop_decay`: the largest loop correction per weight class against physical error rate, showing both the growth with noise and the exactly vanishing class.
3. `plot_correlation_spectrum`: which loop classes carry the injected correlation, correlated model against its marginal-matched twin.
4. `plot_dem_spacetime`: the detector graph laid out on its space and time coordinates, with the injected mechanisms and the loops they close drawn in place. This is the picture to look at if the phrase "the correlated event closes a loop" is not yet concrete.
5. `plot_llr`: the per-shot likelihood-ratio test, with the per-shot error of BP and of the cluster expansion underneath on a log scale.

Note that with the inline backend a figure appears only when its whole cell finishes, so the plots are split one per cell and each recomputes only what it needs.

In [ ]:
# fig 1: convergence vs cluster weight (saved to figures/)
import os
import dem_viz
os.makedirs("figures", exist_ok=True)

curves = {"d=3 R=4, p=0.35": (
    [0, 6, 8, 10],
    [abs(ce.contract_bp() - exact_a) / exact_a]
    + [abs(ce.contract(m) - exact_a) / exact_a for m in (6, 8, 10)],
)}
# fresh d=5 sample: `syn` was overwritten by the correlated-noise cell's loop
dets5_v, obs5_v = circuit5.compile_detector_sampler(seed=0).sample(1, separate_observables=True)
syn5_v = dets5_v[0].astype(int)
tn_b = d5.build(syn5_v, observables={0: int(obs5_v[0][0])})
ex_b = contract_exact(tn_b, optimize=opt5)
ceb = ClusterExpansion(tn_b)
curves["d=5 R=5, p=0.05"] = (
    [0, 6, 8],
    [abs(ceb.contract_bp() - ex_b) / ex_b]
    + [abs(ceb.contract(m) - ex_b) / ex_b for m in (6, 8)],
)
fig = dem_viz.plot_convergence(curves)
fig.savefig("figures/fig1_convergence.png", dpi=150, facecolor=fig.get_facecolor())
fig


In [ ]:
# fig 2: loop decay vs p (saved to figures/). Enumeration stops at weight 8;
# the weight-10 class is identically zero, so nothing is lost by skipping it.
ps, allowed, forbidden = [], [], []
for p in (0.01, 0.05, 0.15, 0.30, 0.45):
    c = stim.Circuit.generated(
        "repetition_code:memory", rounds=3, distance=3,
        before_round_data_depolarization=p, before_measure_flip_probability=p / 2)
    dm = DemTN(c.detector_error_model(decompose_errors=False))
    cep = ClusterExpansion(marginal_tn(dm, np.zeros(dm.num_detectors, dtype=int)))
    by_w = defaultdict(list)
    for F in cep.gen_loops(8):
        by_w[len(F)].append(abs(cep.loop_correction(F)))
    ps.append(p); allowed.append(max(by_w[8])); forbidden.append(max(by_w[6]))
fig = dem_viz.plot_loop_decay(ps, allowed, forbidden)
fig.savefig("figures/fig2_loop_decay.png", dpi=150, facecolor=fig.get_facecolor())
fig


In [ ]:
# figs 3 + 4: correlation spectrum and the space-time loop picture (saved to figures/)
spec_vals, loops_through = {}, []
for name, d in (("corr", dc), ("decorr", dd)):
    cex = ClusterExpansion(marginal_tn(d, syn0))
    by = defaultdict(lambda: [0.0, 0.0])
    for F in cex.gen_loops(8):
        zl = abs(cex.loop_correction(F))
        hot = any(ix.startswith(prefixes) for ix in F)
        by[len(F)][0 if hot else 1] = max(by[len(F)][0 if hot else 1], zl)
        if name == "corr" and hot:
            loops_through.append(F)
    spec_vals[name] = dict(by)
spec = [
    ("|l|=6, through corr. event", spec_vals["corr"][6][0], 0.0),
    ("|l|=6, elsewhere", spec_vals["corr"][6][1], spec_vals["decorr"][6][1]),
    ("|l|=8, through corr. event", spec_vals["corr"][8][0], 0.0),
    ("|l|=8, elsewhere", spec_vals["corr"][8][1], spec_vals["decorr"][8][1]),
]
fig3 = dem_viz.plot_correlation_spectrum(spec)
fig3.savefig("figures/fig3_correlation_spectrum.png", dpi=150, facecolor=fig3.get_facecolor())
fig4 = dem_viz.plot_dem_spacetime(dc, dem_c, corr_ids, loops_through)
fig4.savefig("figures/fig4_spacetime_loops.png", dpi=150, facecolor=fig4.get_facecolor())


In [ ]:
# fig 5: per-shot LLR (saved to figures/)
fig = dem_viz.plot_llr(rows, shot_weights=[int(dets_x[k].sum()) for k in range(len(rows))])
fig.savefig("figures/fig5_llr_per_shot.png", dpi=150, facecolor=fig.get_facecolor())
fig


---

## Surface codes: the network goes three dimensional

Everything so far used the repetition code, whose DEM network is a strip: one space dimension plus time, so contraction width stays bounded as rounds accumulate and exact evaluation is cheap forever. The surface code has two space dimensions plus time. Its DEM network is genuinely three dimensional, and exact contraction cost now grows with the code, which is the regime the whole approximate-contraction program exists for.

Two practical lessons came out of just building these networks, before contracting anything.

**Detector degree explodes, and the dense parity tensor dies first.** With `decompose_errors=False`, a single bulk detector of the $d{=}3$ surface code has up to 48 incident error mechanisms, 71 at $d{=}5$, because every component of every nearby depolarizing channel that flips it counts separately. A dense XOR tensor over 48 legs has $2^{48}$ entries. The first attempt at building this network sat at full CPU for ten minutes for exactly this reason. The chain decomposition in `dem_tn.py` (parity as a running accumulator of degree-3 XORs) makes the same constraint cost $O(k)$, and the network then builds in milliseconds. The lesson generalizes: for circuit-level noise on 2D codes, tensor construction is not free and has to respect degree.

**Backend timing, and an honest verdict on the GPU** (numbers from the benchmark run recorded in `figures/fig6_backend_timing.png`; the timing rows in the next cell are those measurements, not recomputed, since the GPU runs take minutes and one of them cannot run at all). On this Apple machine the GPU backend is torch on Metal, and three independent facts each rule it out for exact DEM contraction. Metal caps tensor rank at 16, and any bond-dimension-2 contraction of width 17 or more must create intermediates of higher rank, so every network past toy size fails with a hard error rather than running slowly. Metal has no float64, so even where it runs the likelihood carries roughly seven digits, fine for choosing an observable, useless for validation targets at the $10^{-12}$ level. And at the sizes that do fit, the GPU is seven times slower than plain numpy, because dispatching a Metal kernel costs a few hundred microseconds while the arithmetic in a dimension-2 tensordot costs almost nothing; we measured the dispatch overhead at 345 times a CPU matmul. Useful GPU acceleration of this workload means CUDA, which has no rank cap and full float64, and that is precisely the niche NVIDIA's CUDA-QX detector-error-model decoder occupies. On this laptop the real levers are torch on CPU (a free 1.3 to 1.4 times) and, above all, better contraction paths: the greedy path gives width 53 for $d{=}5$, $R{=}3$, while hyper-optimized path search is what makes $d{=}5$ at 25 rounds exactly contractible in the Shutty, Newman and Villalonga work, at the price of minutes of one-time search.

**The loop structure changes qualitatively, and this is the physics point.** In the repetition code the short loop classes vanished identically, because its detector graph is bipartite and its only odd-cycle candidates ran through the marginalized observable. The surface code under depolarizing noise has hyperedge mechanisms, weight three and four, and those create genuine short odd cycles: at $d{=}3$, $R{=}3$ we count 225 of 236 loops up to weight 6 carrying nonzero corrections (`figures/fig7_loop_classes.png`). Read that as: the correlations that hyperedges encode, which are exactly the correlations a matching decoder throws away, live at low loop weight in the surface code, so BP corrections start mattering immediately rather than at weight 8. This is both why the cluster expansion has more to do on the surface code and why it has more to offer there.

In [ ]:
# Surface-code figures 6 and 7 (saved to figures/).
# Timing rows are the recorded benchmark measurements (numpy / torch-CPU /
# torch-MPS ms; None = Metal rank-cap failure). The loop-class comparison is
# computed live: it needs only BP plus weight-6 enumeration, a few seconds.
timing_rows = [
    ("d=3 R=1\n(w 6)", 6, 0.4, 0.4, 2.8),
    ("d=3 R=2\n(w 19)", 19, 6.6, 5.9, None),
    ("d=3 R=3\n(w 22)", 22, 46.2, 32.2, None),
    ("d=3 R=5\n(w 22)", 22, 94.8, 72.2, None),
]
fig6 = dem_viz.plot_backend_timing(timing_rows)
fig6.savefig("figures/fig6_backend_timing.png", dpi=150, facecolor=fig6.get_facecolor())


def loop_counts(circuit, maxw):
    dm = DemTN(circuit.detector_error_model(decompose_errors=False))
    cel = ClusterExpansion(marginal_tn(dm, np.zeros(dm.num_detectors, dtype=int)))
    by = defaultdict(lambda: [0, 0])
    for F in cel.gen_loops(maxw):
        by[len(F)][1] += 1
        if abs(cel.loop_correction(F)) > 1e-20:
            by[len(F)][0] += 1
    return {w: tuple(v) for w, v in by.items()}


rep_counts = loop_counts(stim.Circuit.generated(
    "repetition_code:memory", rounds=3, distance=3,
    before_round_data_depolarization=0.05, before_measure_flip_probability=0.02), 8)
surf_counts = loop_counts(stim.Circuit.generated(
    "surface_code:rotated_memory_z", distance=3, rounds=3,
    after_clifford_depolarization=0.003, before_measure_flip_probability=0.003,
    before_round_data_depolarization=0.003, after_reset_flip_probability=0.003), 6)
print("repetition:", rep_counts)
print("surface:   ", surf_counts)
fig7 = dem_viz.plot_loop_class_comparison(rep_counts, surf_counts)
fig7.savefig("figures/fig7_loop_classes.png", dpi=150, facecolor=fig7.get_facecolor())


---

## Shouldn't the X and Z DEMs be split to avoid hyperedges?

The question deserves a careful answer because splitting is the standard move in the decoding literature, and we deliberately do the opposite.

Where splitting comes from: for a CSS code, X errors trigger only Z-type detectors and Z errors trigger only X-type detectors, so under noise with independent X and Z components the DEM factors into two disjoint graphlike halves and each half can be decoded separately with matching. The problem is Y errors and correlated two-qubit gate errors. A Y error is an X and a Z on the same qubit at the same time, so it fires detectors in both sectors at once, and a depolarizing channel produces Y with the same probability as X or Z. In the DEM this shows up as mechanisms whose support spans both sectors, several detectors at a time: the hyperedges. Stim's `decompose_errors=True` exists to serve matching decoders: it factors each hyperedge into graphlike pieces, which restores the two clean graphs at the cost of treating the pieces as independent. That is an approximation with a known accuracy price, because it forgets that the X-sector piece and the Z-sector piece of a Y error always fire together. Correlated-matching decoders and the reweighting tricks in the literature are attempts to claw some of that information back after having thrown it away.

Why we do not split: hyperedges are not a problem for a tensor network, and handling them exactly is the entire advantage this construction has over matching. Concretely, a mechanism that flips $k$ detectors is one Bernoulli tensor and one degree-$(k+1)$ COPY tensor, cost linear in $k$, the same construction as any weight-2 edge. The likelihood the contraction computes then automatically carries the cross-sector correlations: conditioning on the Z-sector detectors that a Y error fired shifts the probabilities of the X-sector detectors it also fired, and the network knows this because a single Bernoulli variable feeds both. Split the model and that conditional information is exactly what is destroyed, and the factorized likelihood $\Pr_X(x_X)\Pr_Z(x_Z)$ is simply the wrong number under depolarizing noise. Shutty, Newman and Villalonga make this point quantitatively: hyperedge structure materially changes the likelihoods, so a differentiable maximum-likelihood pipeline must keep `decompose_errors=False`, as we do everywhere in this notebook.

So what do we do about the costs hyperedges impose, since they are real? Three separate costs, three separate answers. The construction cost, where a bulk detector accumulates dozens of incident mechanisms and dense parity tensors blow up, is solved exactly by the chain decomposition above; note this cost is about how many mechanisms touch a detector, and would arise even in a split model. The contraction-width cost is managed by path optimization, and is the eventual hard wall for any exact method regardless of splitting. And the inference cost, namely that hyperedges create short loops that degrade BP (visible directly in the loop-class figure: 225 nonzero short loops in the surface DEM against effectively none in the repetition DEM), is precisely what the cluster expansion is for. In other words, the standard workflow avoids hyperedges because its decoder cannot represent them; ours keeps them because they carry information, and pays the resulting bill with the loop corrections.

---

## What the network actually looks like, and how a correlated event appears in it

A recurring point of confusion, worth settling with a picture: **an error mechanism is not an edge of the tensor network. It is a node.** Each mechanism contributes a Bernoulli tensor and a COPY tensor, and the COPY is wired by one ordinary edge to each detector parity tensor the mechanism can flip. Detectors are the other node species. Qubits appear nowhere: as Derks et al. put it, the data qubits are compiled away into which detectors each fault flips, so the network's vertex set is mechanisms and detectors, full stop. The figures below (drawn with quimb's own layout engine, saved to `figures/`) show the toy DEM, where the Bernoulli, COPY and XOR chain for each mechanism is individually visible, and the correlated $d{=}4$ repetition DEM with the injected mechanisms in orange: an extra node reaching two detector columns that no ordinary mechanism connects, visibly closing new cycles.

This picture also answers the question of how one runs a loop or cluster expansion on a hypergraph, because at first sight the theory of arXiv:2510.02290 is stated for ordinary graphs and a weight-$k$ mechanism is a $k$-way hyperedge. The answer is that we never run anything on a hypergraph. Turning each hyperedge into a COPY node with $k$ ordinary edges is exactly the classical incidence-graph (Levi graph) construction: hyperedges become vertices, membership becomes edges, and the transformation is exact because a $k$-way shared index and an explicit $k$-leg delta tensor are the same mathematical object. Quimb's hyper routines (`HD1BP` and friends) keep that delta implicit inside a shared index; our builder materializes it as a tensor. Same object, two bookkeepings, but only the materialized form gives the loop expansion what it needs, namely bonds that join exactly two tensors, so that the excitation projector $P^\perp$ is an ordinary operator on a single bond and a loop is an ordinary edge subset. The costs of the conversion are bookkeeping, not approximation: the graph gains one node and $k$ edges per mechanism, and loop weights get renormalized, since a cycle through $k$ mechanisms now uses $2k$ edges. That renormalization is why our loop weights come in the pattern they do, and it is the reading key for the next paragraph.

On the earlier weight-6 versus weight-8 question, one correction to a natural misreading: the relevant "lattice" is not a lattice of qubits and detectors. It is the detector graph, whose vertices are detectors only, with an edge wherever a weight-2 mechanism links two of them; in the repetition code, data errors provide the space edges and measurement errors the time edges, which makes it a grid. Loops in the tensor network are cycles of mechanisms, at twice the edge count. Positions with no detector, meaning the boundaries and the single-detector mechanisms that live there, enter the network as pendant branches: a Bernoulli and COPY hanging off one detector by a single edge. A loop cannot pass through a leaf, so the loop enumeration strips them (the two-core step) before searching, and they contribute to BP but never to loop corrections. The observable node is the one exception to detector-only vertices: it is an extra parity vertex shared by every logical-flipping mechanism, which is precisely how the spurious weight-6 candidates arose and why marginalizing it silenced them.

A word on bond dimension, since anyone coming from many-body physics will ask. **Every bond in these networks has dimension 2**, because every index is one classical bit: a mechanism's firing bit, a running-parity bit inside an XOR chain, or the observable outcome. There is no truncation anywhere and no accuracy knob hiding in the bonds; the network is an exact representation of the distribution at bond dimension 2, which is the opposite of the PEPS situation where the bond dimension is the approximation parameter. The hardness lives elsewhere: in how many dimension-2 legs an intermediate tensor must carry at once, which is the contraction width, so width 22 means a $2^{22}$-entry intermediate spread over 22 separate binary legs. That leg count is exactly what hit Metal's rank-16 cap, where a physics PEPS of the same total size would have packed the information into a few large bonds and sailed under it. The dimension-2 bonds also have one happy consequence we exploited repeatedly: the excitation projector on any bond is rank one, so a loop correction on a cycle factorizes into a product of per-tensor scalars, which is what made the vanishing-loop dissection possible. Where a genuine tunable bond dimension would enter this project is approximate boundary-MPS contraction of the 2+1 dimensional networks, where the compressed boundary bond is the knob; that is the research plan's next stage, not anything in this notebook yet.

In [ ]:
# figs 8 + 9: the tensor network itself, drawn with quimb (saved to figures/)
toy_dem = stim.DetectorErrorModel.from_file(io.StringIO("""
error(0.01) D0
error(0.02) D0 D1
error(0.01) D1
error(0.02) D1 D2
error(0.02) D2
"""))
dt_toy = DemTN(toy_dem)
tn_toy = dt_toy.build(np.zeros(3, dtype=int), observables=None)
fig8 = dem_viz.draw_dem_tn(dt_toy, tn_toy, dem=None, figsize=(7, 5), show_tags=False)
fig8.suptitle("Toy DEM as a tensor network: mechanisms are nodes, not edges",
              fontsize=12, fontweight="bold", x=0.02, ha="left")
fig8.savefig("figures/fig8_tn_structure_toy.png", dpi=150,
             facecolor=fig8.get_facecolor(), bbox_inches="tight")

# the correlated d=4 R=4 DEM from the correlation section, injected events in orange,
# detector tensors pinned to their space-time coordinates
tn_c9 = dc.build(np.zeros(dem_c.num_detectors, dtype=int), observables=None)
tn_c9 |= qtn.Tensor(np.ones(2), inds=("obs0",))
fig9 = dem_viz.draw_dem_tn(dc, tn_c9, dem=dem_c, corr_ids=corr_ids, figsize=(10, 7))
fig9.suptitle("Correlated DEM: the injected mechanism is an extra node wired to two detectors",
              fontsize=12, fontweight="bold", x=0.02, ha="left")
fig9.savefig("figures/fig9_tn_correlated.png", dpi=150,
             facecolor=fig9.get_facecolor(), bbox_inches="tight")


---

## Reproducing the summary figures (`experiments.py`)

The figures used in the slide deck are produced by driver functions in `experiments.py`, so they can be rebuilt from scratch here. The same drivers run from the command line: `python run_surface_ce.py` prints the exact/BP/cluster comparison for a surface code, and `python run_correlated_validation.py` runs the twin-model correlation test. Everything below is seeded, so reruns reproduce the same numbers. The correlated-noise cells default to 100 shots to stay fast; raise `n_shots` for smoother statistics (the deck's phenomenological panel used 2000).

In [ ]:
from experiments import (surface_dem, inject_correlated_pairs, correlated_llr,
                         loop_spectra, fig_repcode_network, fig_llr_combined,
                         fig_fingerprints, fig_dem_zoo, fig_cost_reach)

# the construction figure: every tensor of the two-round d=3 repetition DEM
fig_repcode_network(path="figures/fig12_repcode_slide.png");

In [ ]:
# twin models on the d=3 surface code, phenomenological and circuit level
_, dem_ph, dt_ph = surface_dem(3, 3, 0.01, circuit_level=False)
twins_ph = inject_correlated_pairs(dem_ph, dt_ph)
rows_ph = correlated_llr(twins_ph, n_shots=100)

_, dem_ci, dt_ci = surface_dem(3, 3, 0.003, circuit_level=True)
twins_ci = inject_correlated_pairs(dem_ci, dt_ci)
rows_ci = correlated_llr(twins_ci, n_shots=100, circuit_level=True)

fig_llr_combined(rows_ph, rows_ci, path="figures/fig24_llr_combined.png");

In [ ]:
# loop fingerprints of the same twins, over the d=5 convergence panel
spec_ph = loop_spectra(twins_ph, n_shots=50)
spec_ci = loop_spectra(twins_ci, n_shots=10, circuit_level=True)
fig_fingerprints(spec_ph, spec_ci, path="figures/fig25_fingerprints_combined.png");

In [ ]:
# cost-per-evaluation summary, and the DEM gallery (its BB half needs quits)
fig_cost_reach(path="figures/fig17_cost_reach.png")
fig_dem_zoo(path="figures/fig26_dem_zoo.png");

In [ ]:
# BB [[72,12,6]] under phenomenological noise: BP plus clusters, where
# neither exact contraction (treewidth) nor compressed sweeps (no boundary) run
import numpy as np
from cluster_expansion import ClusterExpansion
from experiments import bb_phenom_dem, build_closed

H, dem_bb, dt_bb = bb_phenom_dem(p=0.01, rounds=3)
rng = np.random.default_rng(11)
e = (rng.random(H.shape[1]) < 0.01).astype(np.uint8)
syn = np.zeros(dem_bb.num_detectors, dtype=np.uint8)
syn[:H.shape[0]] = (H @ e) % 2          # a round-0 error pattern
ce = ClusterExpansion(build_closed(dt_bb, syn))
loops = ce.gen_loops(6)
ce.gen_loops = lambda mw: [F for F in loops if len(F) <= mw]
print(f"{len(loops)} loops to weight 6 | BP logPr "
      f"{np.log(abs(ce.contract_bp())):.4f} | CE M=6 logPr "
      f"{np.log(abs(ce.contract(6))):.4f}")